# SESSION 4 - Build PDF Chatbot

### Task 1: Extract Text from PDF (sample_resume.pdf)

In [ ]:
import PyPDF2

In [ ]:
def extract_pdf_text(filepath):
    with open(filepath, "rb") as f:
        reader = PyPDF2.PdfReader(f)
        extracted_pages = []
        for page in reader.pages:
            page_text = page.extract_text()
            if page_text:
                extracted_pages.append(page_text)
        return "\n".join(extracted_pages)

In [ ]:
resume_text = extract_pdf_text("sample_resume.pdf")
print("Extracted Resume Content:\n")
print(resume_text)

### Task 2: Chunk WhatsApp Chat PDF into 500-Character Chunks

In [ ]:
chat_text = extract_pdf_text("whatsapp_chat.pdf")
print("Total characters extracted from WhatsApp chat:", len(chat_text))

In [ ]:
def chunk_by_characters(text, chunk_size=500):
    chunks = []
    for i in range(0, len(text), chunk_size):
        chunk = text[i:i + chunk_size].strip()
        if chunk:
            chunks.append(chunk)
    return chunks

In [ ]:
chat_chunks = chunk_by_characters(chat_text, chunk_size=500)
print(f"Total 500-character chunks created: {len(chat_chunks)}\n")

for idx, chunk in enumerate(chat_chunks, 1):
    print(f"--- Chunk {idx} ({len(chunk)} characters) ---")
    print(chunk[:140] + "...")
    print()

### Task 3: Generate Embeddings for Chunks (paraphrase-MiniLM-L6-v2)

In [ ]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("paraphrase-MiniLM-L6-v2")

In [ ]:
chunk_embeddings = model.encode(chat_chunks)
print(f"Successfully generated embeddings for {len(chunk_embeddings)} chunks.")
print(f"Embedding dimension size: {chunk_embeddings[0].shape[0]}")

### Task 4: Retrieve Most Relevant Chunk using Cosine Similarity

In [ ]:
from sentence_transformers import util

def find_most_relevant_chunk(query, chunks, embeddings, embed_model):
    query_vector = embed_model.encode(query)
    similarities = util.cos_sim(query_vector, embeddings)[0]
    highest_idx = similarities.argmax().item()
    best_score = similarities[highest_idx].item()
    return chunks[highest_idx], best_score

In [ ]:
user_query = "Who messaged the most?"
best_chunk, score = find_most_relevant_chunk(user_query, chat_chunks, chunk_embeddings, model)

print(f"User Query: '{user_query}'")
print(f"Highest Cosine Similarity Score: {score:.4f}\n")
print("Retrieved Most Relevant Chunk:\n")
print(best_chunk)

### Task 5: Real-World Deployment Ideas for PDF Chatbots in Daily Apps

1. **IRCTC Rail Connect - Dynamic Train Rules & Circulars Bot**
   - **Deployment Idea:** Integrate a PDF RAG chatbot directly into the IRCTC ticket booking portal and app that ingests official railway circulars, Tatkal booking guidelines, refund slabs, and luggage rules.
   - **User Benefit:** Passengers avoid parsing dense, multi-page bureaucratic PDFs. Users can simply ask: *"What are the cancellation charges for an RAC ticket 3 hours before chart preparation?"* and receive immediate, authoritative answers.

2. **Zomato - Restaurant Allergen & Nutrition Spec Assistant**
   - **Deployment Idea:** Deploy a chatbot that indexes restaurant menu PDFs, official ingredient lists, FSSAI hygiene audit certificates, and allergen matrices provided by partner restaurants.
   - **User Benefit:** Diners with severe dietary restrictions (e.g., gluten intolerance, peanut allergy, veganism) can ask: *"Does the mushroom soup contain chicken stock or dairy cream?"* and get verified answers derived directly from the restaurant's kitchen spec documents.

3. **BookMyShow - Event Venue Policy & Layout Guide**
   - **Deployment Idea:** Integrate a venue assistant into event booking pages that indexes venue PDF layout blueprints, parking maps, entry gate instructions, and prohibited items lists for concerts or sports matches.
   - **User Benefit:** Event attendees can ask: *"Are power banks or DSLR cameras allowed inside the Narendra Modi Stadium for Saturday's concert?"* and receive clear, instant answers without scrolling through exhaustive terms-and-conditions documents.